in this notebook we will evaluate the m-e5 model that we are using in parsaqa website

for its semantic search capabilities

using the dedicated libraries that we have developed for the tasks

for searching we will use `ai` library from ai_services repo

fro evaluation we will use `evaluator` library, whihc is a custom library we created specific for reranking search engines


first we install the libraries

In [ ]:
# !pip install git+https://github.com/HadithAi/ai_services.git

In [ ]:
# !https://github.com/HadithAi/evaluator

In [1]:
from elasticsearch import AsyncElasticsearch as Elasticsearch
import tritonclient.http as httpclient
from dotenv import load_dotenv

# our custom libraries
from ai import CosineSemanticSearch
from evaluator.evaluators.search_rank_evaluator import (SearchRankEvaluator,
                                                        LabelStudioRankingTask,
                                                        MetricResults,
                                                        LabelStudio)


import os
load_dotenv()

True

we need triton client for semantic search

In [2]:
triton_client = httpclient.InferenceServerClient(url="172.30.0.113:8000")

In [3]:
# config for the semantic searcher
config = {
    "es_client": Elasticsearch(
            hosts=[{
                'host': os.getenv('ES_URL'),
                'port': 9200,
                'scheme': 'https'
            }],
            basic_auth=(os.getenv('ES_USER'), os.getenv('ES_PASS')),
            verify_certs=False
        ),
    "index_name": "parsaqa_questions",
    "triton_client": triton_client,
    "triton_instruction": """
    retrieve the closest question to this question
    """,
    "triton_model_name": "e5"
    
}

/home/tohidi/miniconda3/envs/general/lib/python3.10/site-packages/elasticsearch/_async/client/__init__.py:403: SecurityWarning: Connecting to 'https://172.30.0.114:9200' using TLS with verify_certs=False is insecure
  _transport = transport_class(


we define the Search Engine here

In [4]:
search_engine = CosineSemanticSearch(config=config)

now we need a list of good questions

that we would want to rerank and evaluate our system based on them

In [5]:

# this the template that we have to create with our data
# they should look like this

# example_data = [
#     {
#         "id": 1,
#         "query": "Top tourist destinations in Japan",
#         "items": [
#             {"id": "a", "body": "Tokyo"},
#             {"id": "b", "body": "Kyoto"},
#             {"id": "c", "body": "Osaka"},
#             {"id": "d", "body": "Hokkaido"},
#             {"id": "e", "body": "Nara"}
#         ]
#     },
#     {
#         "id": 2,
#         "query": "Best deep learning frameworks in 2025",
#         "items": [
#             {"id": "a", "body": "TensorFlow"},
#             {"id": "b", "body": "PyTorch"},
#             {"id": "c", "body": "JAX"},
#             {"id": "d", "body": "MXNet"},
#             {"id": "e", "body": "MindSpore"}
#         ]
#     },
#     {

In [9]:
evaluate_questions = [
    """
    نماز صبح چند رکعت است؟
    """,
    """
    گراز
    """,
    """
    الکل در شکلات
    """,
    """
	خوردن کانگورو چه حکمی داره؟
    """,
	"""
	در قرآن چه مطالبی درباره کره زمین آمده است؟
 	""",
	"""
	حکم بازی های کامپیوتری چیست؟
 	"""
]

In [10]:
questions_results = []

for idx, question in enumerate(evaluate_questions, start=1):
    search_res = await search_engine.search(query=question, size=10, language="fa", source=True)
    
    items = []
    for item_id, hit in zip(search_res['result'], search_res["content"]):
        try:
            question_text = hit["_source"]["question"]["text"]["fa"]
            answer = hit["_source"]['answers'][0][0]['text']['fa']
        except KeyError:
            question = "[متن یافت نشد]"  

        items.append({
            "id": item_id,
            "title": question_text,
            "body": answer
        })
    
    questions_results.append({
        "id": idx,
        "query": question.strip(),
        "items": items
    })

In [11]:
questions_results[:1] # DONE!

[{'id': 1,
  'query': 'نماز صبح چند رکعت است؟',
  'items': [{'id': '3700737818364571',
    'title': 'چرا نمازصبح دو رکعت است؟',
    'body': '\nفلسفه همه احکام و جزئیات آنها به طور تفصیلى روشن نیست و آگاهى از آن دانشى فراتر از تنگناهاى معارف عادى بشرى مى  طلبد. لیکن به طور اجمال، روشن است که همه احکام الهى تابع مصالح و مفاسد واقعى در متعلق آنهاست. بنابراین در صورتى که فلسفه حکمى را بالخصوص ندانیم، بنا بر قاعده کلى فوق باید از آن پیروى کنیم؛ زیرا در آن یقین به وجود مصلحتى هست، هر چند بر ما ناشناخته باشد.حکمت دو رکعتی بودن نماز صبح:اختلاف تعداد رکعات نماز و هم چنین تعیین اوقات و سایر جزئیات عبادات - علاوه بر اسرار و رموزى که در آنها نهفته و ما از آنها اطلاعى نداریم - روحیه تسلیم در برابر خدا و دستورات پیامبر را در ما تقویت و استوار مى  کند.\xa0رکعات نماز که بیشتر ناظر بر بعد عبادی و معنوی و رابطه بین اعمال انسان و هدایت اخروی است عقل کنونی بشر فی حدنفسه ابزاری برای دریافت حکم ارتباط دو رکعتی بودن نماز صبح و آثار آن نسبت به بقیه نماز ها ندارد. در این موارد چاره ای جز تعبد یا قبول حکمت های 

now we have to create the project in label studio using these data

In [ ]:
ls_client = LabelStudio(
    base_url="http://193.134.100.245:8080",
    api_key=os.getenv("LS_API_KEY"),
    timeout=20.0
)

In [ ]:
evaluator = SearchRankEvaluator(ls_client=ls_client)

In [ ]:
project_url = evaluator.create_labelstudio_project(tasks_data=questions_results)
print(f"Label Studio project created: {project_url}")